In [69]:
from dexter.config.constants import Split
from dexter.data.loaders.RetrieverDataset import RetrieverDataset
from dexter.utils.metrics.ExactMatch import ExactMatch
from dexter.retriever.dense.Contriever import Contriever
from dexter.utils.metrics.SimilarityMatch import CosineSimilarity
from dexter.utils.metrics.retrieval.RetrievalMetrics import RetrievalMetrics
from dexter.llms.flant5_engine import FlanT5Engine
import pandas as pd
import joblib

In [70]:
import configparser

config = configparser.ConfigParser()
config['Data-Path'] = {
    'wikimultihopqa' : 'data/',
    'wikimultihopqa-corpus' : 'wiki_musique_corpus.json'
}

with open('config.ini', 'w') as configfile:
    config.write(configfile)

print("config.ini created successfully!")

config.ini created successfully!


In [76]:
loader = RetrieverDataset("wikimultihopqa", "wikimultihopqa-corpus", "config.ini", Split.DEV, tokenizer=None)

Transforming passage dataset: 100%|██████████| 563424/563424 [00:02<00:00, 235405.22it/s]


Harley-Davidson Harley-Davidson
KeysView(<Section: Data-Path>)
12576


100%|██████████| 1200/1200 [03:10<00:00,  6.31it/s]


In [77]:
from dexter.data.datastructures.hyperparameters.dpr import DenseHyperParams

In [78]:
config_instance = DenseHyperParams(query_encoder_path="facebook/contriever", document_encoder_path="facebook/contriever", batch_size=32, show_progress_bar=True)

In [79]:
queries, qrels, corpus = loader.qrels()

In [80]:
contrvr_search = Contriever(config_instance)   
similarity_measure = CosineSimilarity()

C:\Users\San\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\San\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\modeling_utils.py:463: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the 

In [81]:
response = {}

In [17]:
k_values = [1, 3, 5]
for k in k_values:
    response[k] = contrvr_search.retrieve(corpus, queries, 100, similarity_measure, chunk=True, chunksize=40)

token_emb torch.Size([1200, 34, 768])
sentence_emb torch.Size([1200, 768])
token_emb torch.Size([1200, 34, 768])
sentence_emb torch.Size([1200, 768])


  0%|          | 0/40 [00:00<?, ?it/s]

Starting encoding of contexts....


64it [00:01, 42.93it/s]                        


context_embeddings torch.Size([40, 768])


AttributeError: 'Tensor' object has no attribute 'lower'

In [ ]:
joblib.dump(response, "response.pkl")

In [8]:
response = joblib.load("response.pkl")

In [9]:
metric = ExactMatch()

In [ ]:
flant5 = FlanT5Engine(response)

C:\Users\San\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

C:\Users\San\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\San\.cache\huggingface\hub\models--google--flan-t5-xl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [17]:
joblib.dump(flant5, "flant5_response.pkl")

['flant5_responnse.pkl']

In [10]:
flant5 = joblib.load("flant5_response.pkl")

In [41]:
corpus[496418].text()

"2018 Cannes Film Festival Official poster of the 71st Cannes Film Festival featuring Jean - Paul Belmondo and Anna Karina in Pierrot le Fou (1965) Opening film Everybody Knows Closing film The Man Who Killed Don Quixote Location Cannes, France Founded 1946 Awards Palme d'Or (Shoplifters) Hosted by Édouard Baer No. of films 21 (In Competition) 18 (Un Certain Regard) Festival date 8 -- 19 May 2018 Website festival-cannes.com/en"

### Stuff because stuff is weird

qrels gives us relevant documents for each query indexed by query id

responses gives us top-k relevant documents for each query acc to contriever indexed by query id

queries is just the list of questions, use q.text() to get string value, q.id() to get query id

corpus has evidences indexed by the id present in qrels or responses id


In [37]:
metrics = RetrievalMetrics(k_values=[1, 3, 5])
print(metrics.evaluate_retrieval(qrels=qrels, results=response[1]))

({'NDCG@1': 0.43583, 'NDCG@3': 0.20453, 'NDCG@5': 0.14782}, {'MAP@1': 0.04413, 'MAP@3': 0.04413, 'MAP@5': 0.04413}, {'Recall@1': 0.04413, 'Recall@3': 0.04413, 'Recall@5': 0.04413}, {'P@1': 0.43583, 'P@3': 0.14528, 'P@5': 0.08717})


In [45]:
raw_data = loader.base_dataset.raw_data

In [68]:
for r in raw_data:
    print(r.question.text())

Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film)

In [52]:
response[5]

{'13f5ad2c088c11ebbd6fac1f6bf848b6': {'252094': 0.46607813239097595,
  '98709': 0.4693426787853241,
  '59201': 0.4693426787853241,
  '496418': 0.479369193315506,
  '72056': 0.47573965787887573},
 '3057c6c4086111ebbd5dac1f6bf848b6': {'225310': 0.4121391475200653,
  '411824': 0.41234707832336426,
  '205019': 0.42697057127952576,
  '226174': 0.4234209656715393,
  '187876': 0.4499715268611908},
 '89bc944808a111ebbd79ac1f6bf848b6': {'31309': 0.4021065831184387,
  '94595': 0.4167948365211487,
  '357169': 0.40240007638931274,
  '433265': 0.4683046042919159,
  '297135': 0.46830445528030396},
 '633f80660bdd11eba7f7acde48001122': {'123983': 0.4594964385032654,
  '118785': 0.4687654376029968,
  '193327': 0.569388210773468,
  '503936': 0.48808377981185913,
  '211848': 0.47718513011932373},
 '2dc3f9740bda11eba7f7acde48001122': {'104342': 0.45861488580703735,
  '152580': 0.4632769525051117,
  '80533': 0.4811129570007324,
  '475972': 0.47994759678840637,
  '518687': 0.4840882420539856},
 '61df8a820bd

In [61]:
evidences = {}
for q in response[5].keys():
    evidence_ids = list(response[5][q].keys())
    evidence_string_list = []
    for id in evidence_ids:
        evidence_string_list.append(corpus[int(id)].text())
    evidences[q] = evidence_string_list

print(evidences)

{'13f5ad2c088c11ebbd6fac1f6bf848b6': ['Nico Papatakis (5 July 1918 – 17 December 2010) was a Greek- Ethiopian-born naturalised French filmmaker, who lived in France.', 'Maurice Boutel( 1923 at Maghnia in Algeria) is a French film director, screenwriter and dialoguist.', 'Maurice Boutel (1923 at Maghnia in Algeria) is a French film director, screenwriter and dialoguist.', "2018 Cannes Film Festival Official poster of the 71st Cannes Film Festival featuring Jean - Paul Belmondo and Anna Karina in Pierrot le Fou (1965) Opening film Everybody Knows Closing film The Man Who Killed Don Quixote Location Cannes, France Founded 1946 Awards Palme d'Or (Shoplifters) Hosted by Édouard Baer No. of films 21 (In Competition) 18 (Un Certain Regard) Festival date 8 -- 19 May 2018 Website festival-cannes.com/en", 'Philippe Martínez is a French film producer, director, screenwriter, actor, and former President of the Odeon Theater in Marseilles.'], '3057c6c4086111ebbd5dac1f6bf848b6': ['The Falcon\'s Brot

In [66]:
question_df = {"questions":[], "answers":[]}
system_prompt = "Given the question and context, think step by step and output final answer for the question using information in the context and give answer in form of [Final Answer]: \n"
matches = 0
mismatches = 0
ids = []
for row in raw_data:
    if row.question.id() in ids:
        continue
    else:
        ids.append(row.question.id())
    top_3 = " ".join(evidences[row.question.id()][0:5])
    # print(top_3)
    user_prompt = "Given the evidence, Evidence: "+top_3+" \n use the information, think step by step and answer the Question:" + row.question.text()
    # print(user_prompt)
    chain_answer = flant5.get_flant5_completion(system_prompt + " " + user_prompt)
    if "not possible" in chain_answer.lower():
        mismatches+=1
        continue
    elif "unknown" in chain_answer.lower():
        mismatches+=1
        continue
    elif len(chain_answer.split("[Final Answer]:")) >1:
        answer = chain_answer.split("[Final Answer]:")[-1]
        print("************",answer,row.answer.text())
        if row.answer.text().lower() in answer.lower():
            matches+=1
        else:
            mismatches+=1
    else:
        mismatches+=1
        question_df["answers"].append(chain_answer)
        question_df["questions"].append(row.question.text())
        
    final_questions = pd.DataFrame(question_df)
    print("EM", matches/(matches+mismatches))
    print(final_questions)
    final_questions.to_csv("flant5_zero_shot.tsv", sep="\t", index=False)
        

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


EM 0.0
                                           questions answers
0  Are director of film Move (1970 Film) and dire...    [No]
EM 0.0
                                           questions answers
0  Are director of film Move (1970 Film) and dire...    [No]
1  Do both films The Falcon (Film) and Valentin T...    [no]
EM 0.0
                                           questions            answers
0  Are director of film Move (1970 Film) and dire...               [No]
1  Do both films The Falcon (Film) and Valentin T...               [no]
2  Which film whose director is younger, Charge I...  [Charge It To Me]
EM 0.0
                                           questions              answers
0  Are director of film Move (1970 Film) and dire...                 [No]
1  Do both films The Falcon (Film) and Valentin T...                 [no]
2  Which film whose director is younger, Charge I...    [Charge It To Me]
3  What is the date of birth of Mina Gerhardsen's...  [1975-September-14]
EM 0.0
  

KeyboardInterrupt: 